[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb)

# Lab 2: Trace Your Agent

### Instrument the agent with OpenTelemetry and inspect its traces in Phoenix

In notebook 1, you built the agent. Here, OpenInference captures LLM calls automatically,
manual spans make agent and tool steps visible, and every trace goes directly to the shared
Phoenix board on Fly.

The workshop's canonical agent remains in `utils.py`; this notebook attaches a tracer to
that implementation rather than creating a second agent that could drift.


---
## 1. Install the lightweight Phoenix tracing SDKs

Use the OTEL and client SDKs rather than installing a second Phoenix server inside Colab.
The shared server is already running; these packages only export spans and query them.


In [ ]:
%pip install -q arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install packages before importing the agent or tracing SDK. If Colab says the
> runtime needs a restart, restart it now and rerun the notebook from the top.


---
## Bootstrap the workshop agent

Fetch `utils.py` only if needed, load the synthetic ledger, and connect through the same
workshop gateway and seat key as notebook 1.


In [ ]:
import urllib.request
import os

if not os.path.exists("utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils

utils.load_data()
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from utils import NAIVE_SYSTEM_PROMPT, run_agent

print("bootstrap done | seat tag:", SEAT, "| traces will go to the shared Phoenix board")


## 2. Send traces to the shared Phoenix board

Register the shared OTLP collector and authenticate with the board key. The project name
includes your seat tag, so your agent and tool traces are grouped with your seat on Fly.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=f"seat-{SEAT}",
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("Shared Phoenix project:", f"seat-{SEAT}")
print("Phoenix board:", PHOENIX_BOARD_URL)


## 3. Instrument LLM calls and attach the agent tracer

`OpenAIInstrumentor` creates an LLM span for each OpenAI SDK call. The canonical agent
adds the surrounding spans: an **AGENT** root, a **CHAIN** for each router call, a nested
LLM span, then a tool-handling **CHAIN** containing one **TOOL** span per tool execution.
Inputs, outputs, and tool errors are recorded as span attributes.


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(tracer_provider.get_tracer(__name__))
print("OpenAI calls instrumented; manual agent/chain/tool spans enabled")


> **Trace data note:** spans contain prompts and tool inputs/outputs. This workshop uses
> synthetic transactions; do not send personal or otherwise sensitive data to a tracer.


---
## 4. Trace a request the ledger may not support

This request asks for a merchant-location breakdown, but the ledger has no location field.
Run the agent, then use the trace to see which steps it took and what the tool returned.
The exact model response can vary.


In [ ]:
missing_location = run_agent(
    "Show me my spending broken down by merchant location.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/missing-location",
)
print(missing_location["answer"])


### Read the trace in Phoenix

Open the shared Phoenix board and select the `seat-<tag>` project printed in setup. Expand the trace and follow the
parent-child spans. Find the router's LLM call, the selected tool, and the tool's input
and output. A failed tool call should be visible in its TOOL span rather than disappearing
inside a final answer.


In [ ]:
from phoenix.client import Client

phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=f"seat-{SEAT}")
trace_columns = [
    column for column in (
        "name", "span_kind", "parent_id", "status_code",
        "attributes.input.value", "attributes.output.value",
        "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion",
    )
    if column in spans_df.columns
]
display(spans_df[trace_columns].tail(12))


---
## 5. Compare with a successful analysis

Run a second request that asks for net expenses after refunds. In Phoenix, compare its
agent → router/LLM → tool span tree with the missing-information example. Notice how the
same instrumentation captures both paths without changing the tracing code.


In [ ]:
net_expenses = run_agent(
    "Use analyze_transactions to calculate net expenses in October after transaction refunds, excluding income.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/net-expenses-after-refunds",
)
print(net_expenses["answer"])


In [ ]:
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=f"seat-{SEAT}")
display(spans_df[trace_columns].tail(20))


---
## All traces go to the shared board

There is no separate publish step: the missing-information run, the refund analysis, and
any additional agent runs in this runtime are sent directly to your `seat-<tag>` project on
the [shared Phoenix board](https://fmlds-eval-workshop-phoenix.fly.dev).


---
### Continue the workshop

[Next: Notebook 3 — Evaluations](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)
